# YOLO26 × Roboflow BigBossV3 — Google Colab Training Notebook

**Workspace:** `luludapomerania-nddks`  
**Project:** `bigbossv3`  
**Version:** `5`  
**Export:** `yolo26`  
**Model:** pretrained `yolo26n.pt` (object detection)

> This notebook assumes bounding-box object detection. For segmentation, use an appropriate `*-seg.pt` model and segmentation-formatted labels.

### Visual workflow

| 1. Roboflow | → | 2. YOLO dataset | → | 3. YOLO26 weights | → | 4. Train on GPU | → | 5. Validate | → | 6. `best.pt` |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Images + boxes | | `train/`, `valid/`, `data.yaml` | | `yolo26n.pt` | | Adjust neural-network parameters | | mAP, precision, recall | | Saved to Drive |

**Before running:** In Colab, choose **Runtime → Change runtime type → GPU**. Add a Colab secret named `ROBOFLOW_API_KEY` (key icon at left), or enter the key securely when prompted. Then run the cells in order.

## 1. Install dependencies and verify your GPU

`ultralytics` provides the training/validation/prediction API. `roboflow` handles dataset download. GPU training is significantly faster than CPU training.

In [14]:
%pip -q install -U ultralytics roboflow pyyaml ipywidgets

import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
assert torch.cuda.is_available(), "Enable a GPU under Runtime > Change runtime type, then rerun."

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Authenticate to Roboflow securely

Do not put your API key in a publicly shared notebook. Colab Secrets is preferred. If that secret is missing, `getpass()` will prompt for the key without displaying it.

In [16]:
from getpass import getpass
from roboflow import Roboflow

try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = getpass("Enter Roboflow API key: ")

if not API_KEY:
    raise ValueError("A Roboflow API key is required.")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("luludapomerania-nddks").project("bigbossv3")
version = project.version(5)
print("Roboflow project connected: bigbossv3, version 5")

Enter Roboflow API key: ··········
loading Roboflow workspace...
loading Roboflow project...
Roboflow project connected: bigbossv3, version 5


## 3. Download the YOLO26 dataset

`version.download("yolo26")` downloads both the images and their YOLO-format labels. `dataset.location` gives you the extracted dataset folder.

In [19]:
from pathlib import Path

dataset = version.download("yolo26")
DATASET_DIR = Path(dataset.location).resolve()
ORIGINAL_YAML = DATASET_DIR / "data.yaml"

print("Dataset location:", DATASET_DIR)
print("Configuration file:", ORIGINAL_YAML)
assert ORIGINAL_YAML.is_file(), "Roboflow export did not contain data.yaml."


Extracting Dataset Version Zip to bigbossv3-5 in yolo26:: 100%|██████████| 7815/7815 [00:01<00:00, 5424.67it/s]

Dataset location: /content/bigbossv3-5
Configuration file: /content/bigbossv3-5/data.yaml


## 4. Inspect the Roboflow export and prepare training/validation splits

The Roboflow download may contain only `train/images` and `train/labels`. YOLO training needs an independent validation split. This cell first detects existing `valid`, `val`, or `validation` folders. **Only if validation is missing**, and `CREATE_VALIDATION_SPLIT_IF_MISSING=True`, it reserves approximately 20% of training images for validation, **moving** the corresponding images and labels so the exact same files do not appear in both sets. It does not modify anything on Roboflow itself.

> **Research caution:** For final accuracy reporting, prefer a validation split established in Roboflow before augmentations, grouped by video/subject/source where appropriate. Random image-level splits can overestimate performance when near-duplicate images are present. Do not train on test data.


In [20]:
import yaml
import random
import shutil
from pathlib import Path

# Set False to stop and correct the split in Roboflow instead.
CREATE_VALIDATION_SPLIT_IF_MISSING = True
VALID_FRACTION = 0.20
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

with open(ORIGINAL_YAML, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert "names" in config, "Expected class names in data.yaml"

def find_split_dir(root, candidates):
    for name in candidates:
        candidate = root / name / "images"
        if candidate.is_dir():
            return candidate
    return None

train_images = find_split_dir(DATASET_DIR, ["train", "training"])
valid_images = find_split_dir(DATASET_DIR, ["valid", "val", "validation"])
test_images = find_split_dir(DATASET_DIR, ["test", "testing"])

print("Roboflow download:", DATASET_DIR)
print("Subfolders:", sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir()))
print("Original YAML split entries:", {k: config.get(k) for k in ("train", "val", "test")})

assert train_images is not None and train_images.is_dir(), (
    f"No train/images folder found under {DATASET_DIR}"
)

if valid_images is None:
    if not CREATE_VALIDATION_SPLIT_IF_MISSING:
        raise FileNotFoundError(
            "Validation images missing. Create/export a validation split in Roboflow "
            "or set CREATE_VALIDATION_SPLIT_IF_MISSING=True."
        )

    train_files = sorted(p for p in train_images.iterdir()
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    assert len(train_files) >= 5, (
        f"Need at least 5 training images to hold out validation images; found {len(train_files)}."
    )

    n_val = max(1, round(len(train_files) * VALID_FRACTION))
    # Leave at least one training example after splitting.
    n_val = min(n_val, len(train_files) - 1)
    selected = random.Random(42).sample(train_files, n_val)

    valid_images = DATASET_DIR / "valid" / "images"
    valid_labels = DATASET_DIR / "valid" / "labels"
    train_labels = train_images.parent / "labels"
    valid_images.mkdir(parents=True, exist_ok=True)
    valid_labels.mkdir(parents=True, exist_ok=True)

    for img in selected:
        # Move both image and matching YOLO annotation, if present.
        annotation = train_labels / f"{img.stem}.txt"
        shutil.move(str(img), str(valid_images / img.name))
        if annotation.exists():
            shutil.move(str(annotation), str(valid_labels / annotation.name))

    print(f"Created local validation split: {n_val} images moved out of training.")

# Paths explicitly point to the actual images directories.
config["path"] = str(DATASET_DIR)
config["train"] = str(train_images)
config["val"] = str(valid_images)
if test_images is not None and test_images.is_dir():
    config["test"] = str(test_images)
else:
    config.pop("test", None)

DATA_YAML = DATASET_DIR / "colab_data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Corrected YAML path:", DATA_YAML)
print("Classes:", config["names"])

for name, images_dir in (("train", train_images), ("val", valid_images), ("test", test_images)):
    if images_dir is None or not images_dir.is_dir():
        continue
    imgs = [p for p in images_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    labels_dir = images_dir.parent / "labels"
    txts = list(labels_dir.glob("*.txt")) if labels_dir.is_dir() else []
    print(f"{name}: {len(imgs)} images; {len(txts)} label files")
    if name in ("train", "val"):
        assert imgs, f"No images found for {name}: {images_dir}"


Roboflow download: /content/bigbossv3-5
Subfolders: ['train']
Original YAML split entries: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images'}
Created local validation split: 781 images moved out of training.
Corrected YAML path: /content/bigbossv3-5/colab_data.yaml
Classes: ['cleaning_supplies-brush', 'cleaning_supplies-cloth', 'cleaning_supplies-polish', 'cleaning_supplies-sponge', 'drinks-coffee', 'drinks-coke', 'drinks-fanta', 'drinks-kuat', 'drinks-milk', 'drinks-orange_juice', 'foods-broth', 'foods-corn_flour', 'foods-ketchup', 'foods-mayo', 'foods-oats', 'foods-tuna', 'fruits-apple', 'fruits-green_apple', 'fruits-lemon', 'fruits-lime', 'fruits-pear', 'fruits-tangerine', 'snacks-cheese_snack', 'snacks-chocolate_bar', 'snacks-cornflakes', 'snacks-crisps', 'snacks-gum_balls', 'snacks-peanuts', 'snacks-pringles']
train: 3125 images; 3125 label files
val: 781 images; 781 label files


## 5. Mount Google Drive to preserve checkpoints

Colab local storage disappears when a runtime is reset. We save model runs under your Drive. Training writes checkpoints periodically; Drive may be slower than local disk, but it is easier to recover after a disconnect.

In [5]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/YOLO26_BigBossV3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Training output location:", OUTPUT_DIR)

Mounted at /content/drive
Training output location: /content/drive/MyDrive/YOLO26_BigBossV3


## 6. Fine-tune the pretrained YOLO26 Nano model

- `model="yolo26n.pt"`: lightweight pretrained detector; a sensible starting point for Colab GPUs.
- `epochs=80`: maximum training passes over the dataset; early stopping can finish sooner.
- `imgsz=640`: resized image size for training.
- `batch=8`: images per optimizer batch. Reduce to `4` if you see a CUDA out-of-memory error.
- `patience=20`: stop when validation fitness fails to improve for 20 epochs.
- `save_period=10`: keep periodic checkpoint files in addition to best/last.
- `device=0`: use GPU 0.

The output run is named `yolo26n_bigbossv3_v5`.

In [ ]:
model = YOLO("yolo26n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=8,
    device=0,
    patience=20,
    workers=2,
    seed=42,
    plots=True,
    save=True,
    save_period=10,
    project=str(OUTPUT_DIR),
    name="yolo26n_bigbossv3_v5",
    exist_ok=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Training run:", RUN_DIR)
print("Best checkpoint:", BEST_PT)
print("Last checkpoint:", LAST_PT)
assert BEST_PT.exists(), "Training has not produced best.pt"

Ultralytics 8.4.189 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bigbossv3-5/colab_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_bigbossv3_v5, nbs=64, nms=None, opset=None, op

## 7. Visualize training curves

`results.png` summarizes training loss and validation metrics over epochs. A healthy training run normally shows decreasing losses and improving validation metrics; results vary by dataset.

In [ ]:
from IPython.display import Image as NotebookImage, display

plot_path = RUN_DIR / "results.png"
if plot_path.exists():
    display(NotebookImage(filename=str(plot_path)))
else:
    print("Training results.png not found. Look in:", RUN_DIR)

## 8. Validate `best.pt` and print standard detection metrics

- **Precision:** how often reported detections are correct.
- **Recall:** how many annotated objects are found.
- **mAP@0.5:** mean AP at IoU 0.5.
- **mAP@0.5:0.95:** AP averaged across multiple stricter overlap thresholds.

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), split="val", imgsz=640, device=0, plots=True)

print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

## 9. Run detection on a validation image

This visual check shows predicted bounding boxes and labels from your trained model. Keep the test set separate if you plan a final unbiased performance report.

In [ ]:
from PIL import Image as PILImage

candidate_images = sorted(
    p for p in valid_images.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)
assert candidate_images, "No supported validation images were found."

image_path = candidate_images[0]
prediction = best_model.predict(source=str(image_path), conf=0.25, imgsz=640, verbose=False)[0]
print("Prediction source:", image_path.name)
display(PILImage.fromarray(prediction.plot()[..., ::-1]))  # BGR -> RGB

## 10. Interactive visual exploration (optional)

Use the **image selector** and **confidence slider** below to inspect detection quality. Increasing confidence usually suppresses uncertain detections; lowering it often increases the number of boxes.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output

output.enable_custom_widget_manager()

sample_images = candidate_images[:20]
image_dropdown = widgets.Dropdown(
    options=[(p.name, str(p)) for p in sample_images],
    description="Image:"
)
confidence_slider = widgets.FloatSlider(
    value=0.25, min=0.05, max=0.90, step=0.05,
    description="Confidence:", continuous_update=False
)

preview_output = widgets.Output()

def update_preview(*args):
    with preview_output:
        clear_output(wait=True)
        pred = best_model.predict(
            source=image_dropdown.value,
            conf=confidence_slider.value,
            imgsz=640,
            verbose=False,
        )[0]
        display(PILImage.fromarray(pred.plot()[..., ::-1]))

image_dropdown.observe(update_preview, names="value")
confidence_slider.observe(update_preview, names="value")
display(widgets.VBox([image_dropdown, confidence_slider, preview_output]))
update_preview()

## 11. Download trained model weights

Your `best.pt` and `last.pt` are already saved in the Drive run directory. Run this cell **only when you want a browser download**. `best.pt` is normally the checkpoint used for inference.

In [ ]:
from google.colab import files

print("Best weights stored at:", BEST_PT)
# Uncomment the next line to download to your laptop:
# files.download(str(BEST_PT))

## Recovering an interrupted training session

If a run was interrupted and its `last.pt` survived in Google Drive, after mounting Drive and reinstalling the required Python packages, run:

```python
from ultralytics import YOLO
checkpoint = "/content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5/weights/last.pt"
model = YOLO(checkpoint)
model.train(resume=True)
```

Run this snippet manually when recovering. Do **not** run it automatically after successfully finishing training.

### References

- [Ultralytics YOLO26 model documentation](https://docs.ultralytics.com/models/yolo26/)
- [Training mode](https://docs.ultralytics.com/modes/train/)
- [Roboflow integration](https://docs.ultralytics.com/integrations/roboflow/)
- [Object detection dataset specification](https://docs.ultralytics.com/datasets/detect/)




---


**RECOVERY FROM TEMPORARILY STOP**


In [11]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/YOLO26_BigBossV3"
)

RUN_DIR = OUTPUT_DIR / "yolo26n_bigbossv3_v5"

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Previous training directory:", RUN_DIR)
print("Best weights:", BEST_PT)
print("Latest weights:", LAST_PT)

assert LAST_PT.is_file(), "Cannot find last.pt!"

print("Checkpoint found!")

Previous training directory: /content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5
Best weights: /content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5/weights/best.pt
Latest weights: /content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5/weights/last.pt
Checkpoint found!


In [13]:
import shutil

BACKUP_PT = (
    RUN_DIR / "weights" / "last_before_resume.pt"
)

shutil.copy2(LAST_PT, BACKUP_PT)

print("Checkpoint backup:", BACKUP_PT)

Checkpoint backup: /content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5/weights/last_before_resume.pt


In [21]:
from pathlib import Path
from ultralytics import YOLO

# Reconstruct dataset path if cells were not run in order
DATA_YAML = Path("/content/bigbossv3-5/colab_data.yaml")

# Ensure dataset configuration is available
assert DATA_YAML.is_file(), "Dataset YAML not found! Please run the Roboflow login and dataset download cells (Steps 2, 3, and 4) first to recreate it."

# Load previous training state
model = YOLO(str(LAST_PT))

# Resume interrupted training
train_results = model.train(
    resume=True,
    data=str(DATA_YAML),
    device=0,
    workers=0
)

# Retrieve updated checkpoint paths
RUN_DIR = Path(model.trainer.save_dir)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Training run:", RUN_DIR)
print("Best checkpoint:", BEST_PT)
print("Last checkpoint:", LAST_PT)

Ultralytics 8.4.189 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bigbossv3-5/colab_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.000303, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/YOLO26_BigBossV3/yolo26n_bigbossv3_v5/weights/last.pt, momentum=0.937, mosaic=1.0, multi_